In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [ ]:
## initially considering the x = (batch,seq_len,embedding) for the nlp related task
x = torch.zeros(32,100,128)

In [98]:
class LinearAttention(nn.Module):
  def __init__(self,seq_len,embedding_dim,heads,head_dim,num_chunks,chunk_size):
    super().__init__()
    assert seq_len == num_chunks*chunk_size
    assert embedding_dim == heads * head_dim
    self.heads = heads
    self.head_dim = head_dim
    self.chunk_size = chunk_size
    self.num_chunks = num_chunks
    self.Q = nn.Linear(embedding_dim,embedding_dim)
    self.K = nn.Linear(embedding_dim,embedding_dim)
    self.V = nn.Linear(embedding_dim,embedding_dim)

  def feature_map(self,x):
    return torch.clamp(F.elu(x) + 1,min=0.0,max=5.0)

  def forward(self,x):
    # x = (32,100,128)
    B,L,E =  x.shape
    print(type(L))
    q = self.feature_map(self.Q(x)).view(B,L, self.heads, self.head_dim).transpose(1, 2) # (32,100,128) => (32,100,2,64) => (32,2,100,64)
    k = self.feature_map(self.K(x)).view(B,L, self.heads, self.head_dim).transpose(1, 2)
    v = self.feature_map(self.V(x)).view(B,L, self.heads, self.head_dim).transpose(1, 2)
    ## chunk wise parallel
    S_ = []
    Z_ = []
    S =  torch.zeros(B, self.heads, self.head_dim, self.head_dim, device=x.device) # (32,2,64,64)
    Z = torch.zeros(B, self.heads, self.head_dim, 1, device=x.device) # (32,2,64,1)
    for c in range(self.num_chunks):
      ## creating the chunks
      st,ed = (c*self.chunk_size),min((c+1)*self.chunk_size,L)
      k_i = k[:, :, st:ed, :] # (32,2,10,64)
      v_i = v[:, :, st:ed, :] # (32,2,10,64)
      print(k_i.shape,v_i.shape,v_i.transpose(3,2).shape)
      kv = torch.matmul(k_i.transpose(-2,-1),v_i)
      print(kv.shape)
      S = S + kv # (32,2,64,64)
      print(Z.shape,k_i.transpose(-1,-2).shape,k_i.transpose(-1,-2).sum(dim=3,keepdim=True).shape)
      Z = Z + k_i.sum(dim=2).unsqueeze(-1)  # (32,2,64,1) here the embed_dim is add to one and get the scaling at each embed_dim
      print(Z.shape,k_i.sum(dim=2).unsqueeze(-1).shape)
      #break
      S_.append(S.clone())
      Z_.append(Z.clone())

    O = torch.zeros_like(v)
    for c in range(self.num_chunks):
      st,ed = (c*self.chunk_size),min((c+1)*self.chunk_size,L)
      prev_S = S_[c-1] if c > 0 else torch.zeros_like(S)
      prev_Z = Z_[c-1] if c > 0 else torch.zeros_like(Z)
      q_i = q[:, :, st:ed, :]
      num = torch.matmul(q_i,prev_S)
      den = torch.matmul(q_i,prev_Z) + 1e-4
      O[:, :, st:ed, :] = num/den
    return O.transpose(1, 2).reshape(B, L, E)

In [99]:
la = LinearAttention(100,128,2,64,10,10)
la

LinearAttention(
  (Q): Linear(in_features=128, out_features=128, bias=True)
  (K): Linear(in_features=128, out_features=128, bias=True)
  (V): Linear(in_features=128, out_features=128, bias=True)
)

In [100]:
la(x).shape

<class 'int'>
torch.Size([32, 2, 10, 64]) torch.Size([32, 2, 10, 64]) torch.Size([32, 2, 64, 10])
torch.Size([32, 2, 64, 64])
torch.Size([32, 2, 64, 1]) torch.Size([32, 2, 64, 10]) torch.Size([32, 2, 64, 1])
torch.Size([32, 2, 64, 1]) torch.Size([32, 2, 64, 1])
torch.Size([32, 2, 10, 64]) torch.Size([32, 2, 10, 64]) torch.Size([32, 2, 64, 10])
torch.Size([32, 2, 64, 64])
torch.Size([32, 2, 64, 1]) torch.Size([32, 2, 64, 10]) torch.Size([32, 2, 64, 1])
torch.Size([32, 2, 64, 1]) torch.Size([32, 2, 64, 1])
torch.Size([32, 2, 10, 64]) torch.Size([32, 2, 10, 64]) torch.Size([32, 2, 64, 10])
torch.Size([32, 2, 64, 64])
torch.Size([32, 2, 64, 1]) torch.Size([32, 2, 64, 10]) torch.Size([32, 2, 64, 1])
torch.Size([32, 2, 64, 1]) torch.Size([32, 2, 64, 1])
torch.Size([32, 2, 10, 64]) torch.Size([32, 2, 10, 64]) torch.Size([32, 2, 64, 10])
torch.Size([32, 2, 64, 64])
torch.Size([32, 2, 64, 1]) torch.Size([32, 2, 64, 10]) torch.Size([32, 2, 64, 1])
torch.Size([32, 2, 64, 1]) torch.Size([32, 2, 64

torch.Size([32, 100, 128])

In [ ]:
# what lineaer attention we had considered
# S,Z = 0
# S = S + fu(K) @ fu(V)_t
# Z = Z + fu(k)
# attention = ( fu(q)_t @ S ) / (fu(q)_t @ Z)

# ---------------------------------
# k.v_t = (32,100,128) @ (32,128,100) = (32,100,100)